Prachee Prasad, 381060

Lab 3: Implement a Federated Learning framework for a linear regression task where multiple clients collaboratively learn a global prediction model without sharing their raw data. For a house price dataset can be split across clients, each client trains a local linear regression model, and a central server applies Federated Averaging (FedAvg) to combine the local model parameters into a global model while preserving data privacy.

In [2]:
import pandas as pd
import numpy as np
import csv
from pathlib import Path

# Load dataset
data_path = Path("/content/Housing.xls")
df = pd.read_csv(data_path, sep=",", engine="python", quoting=csv.QUOTE_NONE)

# Clean column names and string cells
(df.columns) = [c.strip().strip('"').strip("'") for c in df.columns]
for col in df.select_dtypes(include="object").columns:
    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
        .str.strip('"')
        .str.strip("'")
        .str.lower()
    )

# Encode binary yes/no fields
binary_cols = [
    "mainroad",
    "guestroom",
    "basement",
    "hotwaterheating",
    "airconditioning",
    "prefarea",
]
for col in binary_cols:
    if col not in df.columns:
        raise KeyError(f"Expected column '{col}' not found after parsing.")
    df[col] = df[col].map({"yes": 1, "no": 0})

# One-hot encode furnishing status
if "furnishingstatus" in df.columns:
    df = pd.get_dummies(df, columns=["furnishingstatus"], drop_first=True)

# Features/target
if "price" not in df.columns:
    raise KeyError("Expected target column 'price' not found after parsing.")
feature_cols = [c for c in df.columns if c != "price"]
X = df[feature_cols].astype(float).values
y = df["price"].astype(float).values

# Normalize features for stable training
X_mean, X_std = X.mean(axis=0), X.std(axis=0) + 1e-8
X = (X - X_mean) / X_std

print(f"Loaded {len(df)} samples with {X.shape[1]} features.")

Loaded 545 samples with 13 features.


In [3]:
class Client:
    def __init__(self, client_id: int, x: np.ndarray, y: np.ndarray, lr: float = 0.05, local_epochs: int = 20):
        self.client_id = client_id
        self.x = x
        self.y = y
        self.lr = lr
        self.local_epochs = local_epochs

    def train(self, global_weights: np.ndarray) -> np.ndarray:
        w = global_weights.copy()
        n = len(self.y)
        for _ in range(self.local_epochs):
            preds = self.x @ w
            grad = (self.x.T @ (preds - self.y)) / n
            w -= self.lr * grad
        return w


class Server:
    def __init__(self, num_features: int):
        self.global_weights = np.zeros(num_features)

    def aggregate(self, client_weights: list[np.ndarray]) -> None:
        self.global_weights = np.mean(client_weights, axis=0)

    def evaluate_rmse(self, x: np.ndarray, y: np.ndarray) -> float:
        preds = x @ self.global_weights
        return float(np.sqrt(np.mean((preds - y) ** 2)))

In [4]:
def split_clients(x: np.ndarray, y: np.ndarray, num_clients: int) -> list[Client]:
    # Shuffle before splitting to avoid ordered bias
    idx = np.random.permutation(len(y))
    x_shuff, y_shuff = x[idx], y[idx]
    splits = np.array_split(np.arange(len(y)), num_clients)
    clients = []
    for cid, inds in enumerate(splits):
        clients.append(Client(cid, x_shuff[inds], y_shuff[inds]))
    return clients


def run_federated_training(num_rounds: int = 30, num_clients: int = 5) -> tuple[np.ndarray, list[float]]:
    clients = split_clients(X, y, num_clients=num_clients)
    server = Server(num_features=X.shape[1])

    history = []
    for round_idx in range(1, num_rounds + 1):
        client_weights = [c.train(server.global_weights) for c in clients]
        server.aggregate(client_weights)
        rmse = server.evaluate_rmse(X, y)
        history.append(rmse)
        print(f"Round {round_idx:02d} | Global RMSE: {rmse:,.0f}")
    return server.global_weights, history

In [5]:
# Run federated training
final_weights, rmse_history = run_federated_training(num_rounds=25, num_clients=6)

print("\nFinal global weights shape:", final_weights.shape)
print("First 5 weights:", final_weights[:5])

Round 01 | Global RMSE: 4,891,224
Round 02 | Global RMSE: 4,885,000
Round 03 | Global RMSE: 4,884,272
Round 04 | Global RMSE: 4,884,157
Round 05 | Global RMSE: 4,884,164
Round 06 | Global RMSE: 4,884,195
Round 07 | Global RMSE: 4,884,224
Round 08 | Global RMSE: 4,884,247
Round 09 | Global RMSE: 4,884,264
Round 10 | Global RMSE: 4,884,275
Round 11 | Global RMSE: 4,884,283
Round 12 | Global RMSE: 4,884,288
Round 13 | Global RMSE: 4,884,292
Round 14 | Global RMSE: 4,884,294
Round 15 | Global RMSE: 4,884,295
Round 16 | Global RMSE: 4,884,296
Round 17 | Global RMSE: 4,884,297
Round 18 | Global RMSE: 4,884,298
Round 19 | Global RMSE: 4,884,298
Round 20 | Global RMSE: 4,884,298
Round 21 | Global RMSE: 4,884,298
Round 22 | Global RMSE: 4,884,298
Round 23 | Global RMSE: 4,884,298
Round 24 | Global RMSE: 4,884,298
Round 25 | Global RMSE: 4,884,298

Final global weights shape: (13,)
First 5 weights: [443455.00197099  17131.97278833 495157.06751396 439302.72409468
 199595.66836113]
